# Week 04 — Day 22: Data QA completeness checks on real Slovenian power data

Goal: verify whether the curated datasets are complete, consistent, and safe for analysis.

This notebook checks:
- file loading and timestamp parsing,
- dataset coverage,
- duplicate timestamps,
- null values,
- missing intervals,
- DST behavior,
- settlement-type structure,
- value ranges,
- known DA + imbalance timestamp mismatches.

In [3]:
# Import core libraries for data loading, timestamp checks, and QA summaries.
from pathlib import Path
import pandas as pd
import numpy as np

In [4]:
# Set display options so QA tables are easier to inspect in the notebook.
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

In [5]:
# Define the curated data folder.
# This notebook assumes all curated files are stored in the top-level data/curated folder.

DATA_DIR = Path("../data/curated")

# Define all datasets used in Day 22 QA.
# The config keeps the notebook reusable: if a new Slovenian power dataset is added later,
# add it here with its file name, timestamp column, and basic QA rules.

DATASET_CONFIG = {
    "da_2025": {
        "file": DATA_DIR / "da_prices_si_2025_clean.csv",
        "timestamp_col": "timestamp",
        "price_columns": ["price_eur_mwh"],
        "important_columns": ["price_eur_mwh"],
        "mtu_rules": [
            {"start": None, "end": "2025-10-01", "mtu_minutes": 60},
            {"start": "2025-10-01", "end": None, "mtu_minutes": 15},
        ],
        "interval_count_method": "rows",
    },
    "da_2026": {
        "file": DATA_DIR / "da_prices_si_clean.csv",
        "timestamp_col": "timestamp",
        "price_columns": ["price_eur_mwh"],
        "important_columns": ["price_eur_mwh"],
        "mtu_rules": [
            {"start": None, "end": None, "mtu_minutes": 15},
        ],
        "interval_count_method": "rows",
    },
    "imbalance": {
        "file": DATA_DIR / "imbalance_si_clean.csv",
        "timestamp_col": "timestamp",
        "price_columns": ["imbalance_price_pos", "imbalance_price_neg", "sipx"],
        "important_columns": ["imbalance_price_pos", "imbalance_price_neg", "sipx", "settlement_type"],
        "mtu_rules": [
            {"start": None, "end": None, "mtu_minutes": 15},
        ],
        "interval_count_method": "rows",
    },
    "balancing": {
        "file": DATA_DIR / "balancing_volumes_si_clean.csv",
        "timestamp_col": "timestamp",
        "price_columns": ["aFRR EUR/MWh", "mFRR EUR/MWh", "VoAA+ EUR/MWh", "VoAA- EUR/MWh"],
        "important_columns": ["Wpos MWh", "Wneg MWh", "Spos EUR", "Sneg EUR", "settlement_type"],
        "mtu_rules": [
            {"start": None, "end": None, "mtu_minutes": 15},
        ],
        # Balancing may contain more than one row per timestamp because of settlement_type.
        # For interval completeness, unique timestamps are more useful than raw row count.
        "interval_count_method": "unique_timestamps",
    },
}

## QA context

The Slovenian DA price data has an important market-structure feature:

- before 2025-10-01: 60-minute MTU,
- from 2025-10-01 onward: 15-minute MTU.

This mixed resolution in the 2025 DA file is expected and is not treated as a data-quality problem.

The Borzen datasets contain settlement data and may require checking `timestamp + settlement_type`, not timestamp alone.

DST expectations:

| Day type | 60-min MTU | 15-min MTU |
|---|---:|---:|
| Normal day | 24 | 96 |
| Spring DST day | 23 | 92 |
| Autumn DST day | 25 | 100 |

In [6]:
# Define helper functions for Slovenian power-market QA.
# These functions make the notebook reusable for other local power datasets.

def last_sunday(year: int, month: int) -> pd.Timestamp:
    """Return the last Sunday of a given month."""
    last_day = pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)
    return last_day - pd.Timedelta(days=(last_day.weekday() + 1) % 7)


def is_spring_dst_day(date_value) -> bool:
    """Check whether a date is the spring DST transition day in Europe/Ljubljana."""
    date_value = pd.Timestamp(date_value)
    return date_value.normalize() == last_sunday(date_value.year, 3).normalize()


def is_autumn_dst_day(date_value) -> bool:
    """Check whether a date is the autumn DST transition day in Europe/Ljubljana."""
    date_value = pd.Timestamp(date_value)
    return date_value.normalize() == last_sunday(date_value.year, 10).normalize()


def dst_day_type(date_value) -> str:
    """Classify a date as normal, spring DST, or autumn DST."""
    if is_spring_dst_day(date_value):
        return "spring_dst_day"
    if is_autumn_dst_day(date_value):
        return "autumn_dst_day"
    return "normal_day"


def expected_market_intervals(date_value, mtu_minutes: int) -> int:
    """Return expected number of market intervals for a date and MTU."""
    normal_intervals = int(24 * 60 / mtu_minutes)
    intervals_per_hour = int(60 / mtu_minutes)

    if is_spring_dst_day(date_value):
        return normal_intervals - intervals_per_hour

    if is_autumn_dst_day(date_value):
        return normal_intervals + intervals_per_hour

    return normal_intervals


def assign_mtu_minutes(df: pd.DataFrame, mtu_rules: list, timestamp_col: str = "timestamp") -> pd.Series:
    """Assign MTU minutes to each row based on timestamp and configured MTU rules."""
    mtu = pd.Series(index=df.index, dtype="float64")

    for rule in mtu_rules:
        start = pd.Timestamp(rule["start"]) if rule["start"] is not None else None
        end = pd.Timestamp(rule["end"]) if rule["end"] is not None else None
        mtu_minutes = rule["mtu_minutes"]

        mask = pd.Series(True, index=df.index)

        if start is not None:
            mask = mask & (df[timestamp_col] >= start)

        if end is not None:
            mask = mask & (df[timestamp_col] < end)

        mtu.loc[mask] = mtu_minutes

    return mtu.astype("Int64")


def expected_timestamp_occurrences_for_day(date_value, mtu_minutes: int) -> pd.DataFrame:
    """
    Build expected timestamp labels for one delivery day.

    For spring DST, the 02:00 hour is skipped.
    For autumn DST, the 02:00 hour appears twice in local time.
    """
    date_value = pd.Timestamp(date_value).normalize()
    freq = f"{mtu_minutes}min"

    base_times = pd.date_range(
        start=date_value,
        end=date_value + pd.Timedelta(days=1),
        freq=freq,
        inclusive="left"
    )

    expected_times = pd.DataFrame({"timestamp": base_times})

    if is_spring_dst_day(date_value):
        expected_times = expected_times[expected_times["timestamp"].dt.hour != 2]

    if is_autumn_dst_day(date_value):
        repeated_hour = expected_times[expected_times["timestamp"].dt.hour == 2].copy()
        expected_times = pd.concat([expected_times, repeated_hour], ignore_index=True)

    expected_counts = (
        expected_times
        .value_counts("timestamp")
        .reset_index(name="expected_occurrences")
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    return expected_counts


def classify_completeness_reason(date_value, interval_count: int, expected_count: int) -> str:
    """Create a readable completeness label for a delivery day."""
    day_type = dst_day_type(date_value)

    if interval_count == expected_count:
        return f"complete_{day_type}"

    if interval_count < expected_count:
        return f"incomplete_{day_type}"

    return f"extra_intervals_{day_type}"

In [7]:
# Check that all configured files exist before loading.
# This prevents silent path or filename errors.

for dataset_name, config in DATASET_CONFIG.items():
    file_path = config["file"]
    print(dataset_name, "| exists:", file_path.exists(), "|", file_path)

da_2025 | exists: True | ..\data\curated\da_prices_si_2025_clean.csv
da_2026 | exists: True | ..\data\curated\da_prices_si_clean.csv
imbalance | exists: True | ..\data\curated\imbalance_si_clean.csv
balancing | exists: True | ..\data\curated\balancing_volumes_si_clean.csv


In [8]:
# Load each curated CSV into a separate dataframe.
# Each dataset is kept separate because Day 22 QA must identify issues per source file.

datasets = {}

for dataset_name, config in DATASET_CONFIG.items():
    file_path = config["file"]
    timestamp_col = config["timestamp_col"]

    df = pd.read_csv(file_path, parse_dates=[timestamp_col])

    # Standardize timestamp column name to "timestamp" for reusable QA functions.
    if timestamp_col != "timestamp":
        df = df.rename(columns={timestamp_col: "timestamp"})

    df = df.sort_values("timestamp").reset_index(drop=True)

    datasets[dataset_name] = df

    print(f"{dataset_name}: {df.shape[0]} rows, {df.shape[1]} columns")

da_2025: 15387 rows, 2 columns
da_2026: 8541 rows, 2 columns
imbalance: 11520 rows, 5 columns
balancing: 20160 rows, 66 columns


In [9]:
# Show columns, data types, first timestamp, and last timestamp for each dataset.
# This confirms whether timestamp parsing worked correctly and whether the files are readable.

for dataset_name, df in datasets.items():
    print("=" * 100)
    print(dataset_name)
    print("- rows/columns:", df.shape)
    print("- first timestamp:", df["timestamp"].min())
    print("- last timestamp:", df["timestamp"].max())
    print("- columns:", list(df.columns))
    print("- dtypes:")
    print(df.dtypes)
    print("- first 3 rows:")
    display(df.head(3))

da_2025
- rows/columns: (15387, 2)
- first timestamp: 2025-01-01 00:00:00
- last timestamp: 2026-01-01 00:00:00
- columns: ['timestamp', 'price_eur_mwh']
- dtypes:
timestamp        datetime64[us]
price_eur_mwh           float64
dtype: object
- first 3 rows:


,timestamp,price_eur_mwh
0,2025-01-01 00:00:00,118.46
1,2025-01-01 01:00:00,129.07
2,2025-01-01 02:00:00,121.10


da_2026
- rows/columns: (8541, 2)
- first timestamp: 2026-01-01 00:00:00
- last timestamp: 2026-04-01 00:00:00
- columns: ['timestamp', 'price_eur_mwh']
- dtypes:
timestamp        datetime64[us]
price_eur_mwh           float64
dtype: object
- first 3 rows:


,timestamp,price_eur_mwh
0,2026-01-01 00:00:00,100.51
1,2026-01-01 00:15:00,94.03
2,2026-01-01 00:30:00,92.01


imbalance
- rows/columns: (11520, 5)
- first timestamp: 2025-11-01 00:15:00
- last timestamp: 2026-03-01 00:00:00
- columns: ['timestamp', 'imbalance_price_pos', 'imbalance_price_neg', 'sipx', 'settlement_type']
- dtypes:
timestamp              datetime64[us]
imbalance_price_pos           float64
imbalance_price_neg           float64
sipx                          float64
settlement_type                   str
dtype: object
- first 3 rows:


,timestamp,imbalance_price_pos,imbalance_price_neg,sipx,settlement_type
0,2025-11-01 00:15:00,20.71,20.71,103.63,Second imbalance settlement
1,2025-11-01 00:30:00,20.33,20.33,106.03,Second imbalance settlement
2,2025-11-01 00:45:00,20.94,20.94,85.55,Second imbalance settlement


balancing
- rows/columns: (20160, 66)
- first timestamp: 2025-11-01 00:15:00
- last timestamp: 2026-03-01 00:00:00
- columns: ['timestamp', 'Wpos MWh', 'Wneg MWh', 'Spos EUR', 'Sneg EUR', "WpozFCR_Q' MWh", "WnegFCR_Q' MWh", 'aRPF+akt (aFRR+act) MWh MWh', 'aFRR MWh', 'aFRR EUR', 'aFRR EUR/MWh', 'aRPF-akt (aFRR-act) MWh MWh', 'naFRR MWh', 'naFRR EUR', 'naFRR EUR/MWh', 'mFRR MWh', 'mFRR EUR', 'mFRR EUR/MWh', 'nmFRR  MWh', 'nmFRR  EUR', 'nmFRR  EUR/MWh', 'RR MWh', 'RR EUR', 'RR EUR/MWh', 'nRR MWh', 'nRR EUR', 'nRR EUR/MWh', 'IGCC (pos) MWh', 'IGCC (pos) EUR', 'IGCC (pos) EUR/MWh', 'IGCC (neg) MWh', 'IGCC (neg) EUR', 'IGCC (neg) EUR/MWh', 'VoAA+ EUR/MWh', 'VoAA- EUR/MWh', 'WnegFSkar MWh', 'FSkar (pos) MWh', 'SnegFSkar EUR', 'SpozFSkar EUR', 'Inbalance BS W+ MWh', 'Inbalance BS W- MWh', 'aFRR+ akt TUJ>SI EUR EUR', 'aFRR- akt TUJ>SI EUR EUR', 'aFRR+ akt TUJ>SII MWh MWh', 'aFRR- akt TUJ>SI MWh MWh', 'aFRR+ akt SI>SI EUR EUR', 'aFRR- akt SI>SI EUR EUR', 'aFRR+ akt SI>SI MWh MWh', 'aFRR- akt SI>

,timestamp,Wpos MWh,Wneg MWh,Spos EUR,Sneg EUR,WpozFCR_Q' MWh,WnegFCR_Q' MWh,aRPF+akt (aFRR+act) MWh MWh,aFRR MWh,aFRR EUR,aFRR EUR/MWh,aRPF-akt (aFRR-act) MWh MWh,naFRR MWh,naFRR EUR,naFRR EUR/MWh,mFRR MWh,mFRR EUR,mFRR EUR/MWh,nmFRR MWh,nmFRR EUR,nmFRR EUR/MWh,RR MWh,RR EUR,RR EUR/MWh,nRR MWh,nRR EUR,nRR EUR/MWh,IGCC (pos) MWh,IGCC (pos) EUR,IGCC (pos) EUR/MWh,IGCC (neg) MWh,IGCC (neg) EUR,IGCC (neg) EUR/MWh,VoAA+ EUR/MWh,VoAA- EUR/MWh,WnegFSkar MWh,FSkar (pos) MWh,SnegFSkar EUR,SpozFSkar EUR,Inbalance BS W+ MWh,Inbalance BS W- MWh,aFRR+ akt TUJ>SI EUR EUR,aFRR- akt TUJ>SI EUR EUR,aFRR+ akt TUJ>SII MWh MWh,aFRR- akt TUJ>SI MWh MWh,aFRR+ akt SI>SI EUR EUR,aFRR- akt SI>SI EUR EUR,aFRR+ akt SI>SI MWh MWh,aFRR- akt SI>SI MWh MWh,mFRR+ akt TUJ>SI EUR EUR,mFRR- akt TUJ>SI EUR EUR,mFRR+ akt TUJ>SI MWh MWh,mFRR- akt TUJ>SI MWh MWh,mFRR+ akt SI>SI EUR EUR,mFRR- akt SI>SI EUR EUR,mFRR+ akt SI>SI MWh MWh,mFRR- akt SI>SI MWh MWh,QWpozVTL_NOS_Q MWh,QWnegVTL_NOS_Q MWh,QWpozVTL_HOPS_Q MWh,QWnegVTL_HOPS_Q MWh,QWpozVTL_PICASSO_Q MWh,QWnegVTL_PICASSO_Q MWh,QWpozVTL_MARI_Q MWh,QWnegVTL_MARI_Q MWh,settlement_type
0,2025-11-01 00:15:00,0.010,-20.263,1.54,-795.50,0.041,-0.002,0.01,0.01,1.54,154.0,-1.045,-0.918,-19.01,20.71,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00,0.00,-17.111,-734.57,42.93,0.0,0.0,-2.234,0.000,-41.92,0.00,27.106,-6.763,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
1,2025-11-01 00:30:00,0.200,-22.249,12.49,-798.75,0.011,-0.012,0.00,0.00,0.00,0.0,-2.589,-2.511,-51.06,20.33,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.00,0.00,-19.001,-737.58,38.82,0.0,0.0,-0.737,0.200,-10.11,12.49,24.635,-2.507,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2,2025-11-01 00:45:00,1.119,-10.075,61.84,-281.16,0.018,-0.012,0.00,0.00,0.00,0.0,-0.853,-0.780,-16.33,20.94,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.045,2.21,49.11,-9.093,-249.87,27.48,0.0,0.0,-0.202,1.074,-14.96,59.63,13.439,-4.415,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement


In [10]:
# Create a compact QA summary for each dataset.
# This checks coverage, duplicate timestamps, and total null values.

coverage_summary = []

for dataset_name, df in datasets.items():
    coverage_summary.append({
        "dataset": dataset_name,
        "rows": len(df),
        "columns": len(df.columns),
        "first_timestamp": df["timestamp"].min(),
        "last_timestamp": df["timestamp"].max(),
        "duplicate_timestamps": df["timestamp"].duplicated().sum(),
        "total_null_values": df.isna().sum().sum(),
    })

coverage_summary_df = pd.DataFrame(coverage_summary)

display(coverage_summary_df)

,dataset,rows,columns,first_timestamp,last_timestamp,duplicate_timestamps,total_null_values
0,da_2025,15387,2,2025-01-01 00:00:00,2026-01-01,4,0
1,da_2026,8541,2,2026-01-01 00:00:00,2026-04-01,0,0
2,imbalance,11520,5,2025-11-01 00:15:00,2026-03-01,0,0
3,balancing,20160,66,2025-11-01 00:15:00,2026-03-01,8640,0


## Initial QA coverage finding

The curated datasets were loaded successfully and timestamps were parsed as datetime.

This first QA table checks:
- row count,
- column count,
- time coverage,
- duplicate timestamps,
- total null values.

This does not yet prove full completeness. It only confirms that the datasets are structurally usable for deeper QA checks.

In [11]:
# Check duplicate timestamps and duplicate timestamp + settlement_type combinations.
# This separates true duplicate problems from expected settlement-data structure.

duplicate_summary = []

for dataset_name, df in datasets.items():
    row = {
        "dataset": dataset_name,
        "rows": len(df),
        "duplicate_timestamps": df["timestamp"].duplicated().sum(),
    }

    if "settlement_type" in df.columns:
        row["duplicate_timestamp_settlement_type"] = df.duplicated(
            subset=["timestamp", "settlement_type"]
        ).sum()
    else:
        row["duplicate_timestamp_settlement_type"] = np.nan

    duplicate_summary.append(row)

duplicate_summary_df = pd.DataFrame(duplicate_summary)

display(duplicate_summary_df)

,dataset,rows,duplicate_timestamps,duplicate_timestamp_settlement_type
0,da_2025,15387,4,NaN
1,da_2026,8541,0,NaN
2,imbalance,11520,0,0.0
3,balancing,20160,8640,8640.0


In [12]:
# Inspect duplicate timestamp rows only where duplicates exist.
# For DA 2025, duplicates are likely related to the autumn DST repeated hour.
# For balancing, duplicates may be explained by settlement_type.

for dataset_name, df in datasets.items():
    duplicate_rows = (
        df[df["timestamp"].duplicated(keep=False)]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    if len(duplicate_rows) > 0:
        print("=" * 100)
        print(dataset_name)
        print("Duplicate timestamp rows:", len(duplicate_rows))
        print("Unique duplicated timestamps:", duplicate_rows["timestamp"].nunique())
        display(duplicate_rows.head(40))

da_2025
Duplicate timestamp rows: 8
Unique duplicated timestamps: 4


,timestamp,price_eur_mwh
0,2025-10-26 02:00:00,75.33
1,2025-10-26 02:00:00,63.05
2,2025-10-26 02:15:00,67.41
3,2025-10-26 02:15:00,65.54
4,2025-10-26 02:30:00,60.01
5,2025-10-26 02:30:00,63.93
6,2025-10-26 02:45:00,53.67
7,2025-10-26 02:45:00,58.91


balancing
Duplicate timestamp rows: 17280
Unique duplicated timestamps: 8640


,timestamp,Wpos MWh,Wneg MWh,Spos EUR,Sneg EUR,WpozFCR_Q' MWh,WnegFCR_Q' MWh,aRPF+akt (aFRR+act) MWh MWh,aFRR MWh,aFRR EUR,aFRR EUR/MWh,aRPF-akt (aFRR-act) MWh MWh,naFRR MWh,naFRR EUR,naFRR EUR/MWh,mFRR MWh,mFRR EUR,mFRR EUR/MWh,nmFRR MWh,nmFRR EUR,nmFRR EUR/MWh,RR MWh,RR EUR,RR EUR/MWh,nRR MWh,nRR EUR,nRR EUR/MWh,IGCC (pos) MWh,IGCC (pos) EUR,IGCC (pos) EUR/MWh,IGCC (neg) MWh,IGCC (neg) EUR,IGCC (neg) EUR/MWh,VoAA+ EUR/MWh,VoAA- EUR/MWh,WnegFSkar MWh,FSkar (pos) MWh,SnegFSkar EUR,SpozFSkar EUR,Inbalance BS W+ MWh,Inbalance BS W- MWh,aFRR+ akt TUJ>SI EUR EUR,aFRR- akt TUJ>SI EUR EUR,aFRR+ akt TUJ>SII MWh MWh,aFRR- akt TUJ>SI MWh MWh,aFRR+ akt SI>SI EUR EUR,aFRR- akt SI>SI EUR EUR,aFRR+ akt SI>SI MWh MWh,aFRR- akt SI>SI MWh MWh,mFRR+ akt TUJ>SI EUR EUR,mFRR- akt TUJ>SI EUR EUR,mFRR+ akt TUJ>SI MWh MWh,mFRR- akt TUJ>SI MWh MWh,mFRR+ akt SI>SI EUR EUR,mFRR- akt SI>SI EUR EUR,mFRR+ akt SI>SI MWh MWh,mFRR- akt SI>SI MWh MWh,QWpozVTL_NOS_Q MWh,QWnegVTL_NOS_Q MWh,QWpozVTL_HOPS_Q MWh,QWnegVTL_HOPS_Q MWh,QWpozVTL_PICASSO_Q MWh,QWnegVTL_PICASSO_Q MWh,QWpozVTL_MARI_Q MWh,QWnegVTL_MARI_Q MWh,settlement_type
0,2025-12-01 00:15:00,2.287,-18.030,208.94,-633.84,0.000,0.000,0.000,0.000,0.00,0.00,-2.928,-2.721,-91.37,33.58,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.039,1.31,33.59,-11.415,-528.06,46.26,0.0,0.0,-3.894,2.248,-14.41,207.63,22.106,-6.180,0.0,0.0,0.0,0.0,0.00,-91.37,0.000,-2.721,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
1,2025-12-01 00:15:00,2.287,-18.030,208.94,-633.84,0.000,0.000,0.000,0.000,0.00,0.00,-2.928,-2.721,-91.37,33.58,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.039,1.31,33.59,-11.415,-528.06,46.26,0.0,0.0,-3.894,2.248,-14.41,207.63,22.106,-6.180,0.0,0.0,0.0,0.0,0.00,-91.37,0.000,-2.721,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2,2025-12-01 00:30:00,2.405,-3.077,129.54,-228.36,0.000,0.000,0.065,0.065,10.07,154.92,-0.164,-0.164,-5.54,33.78,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.708,23.91,33.77,-2.897,-221.34,76.40,0.0,0.0,-0.016,1.632,-1.48,95.56,7.807,-7.115,0.0,0.0,0.0,0.0,10.07,-5.54,0.065,-0.164,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
3,2025-12-01 00:30:00,2.405,-3.077,129.54,-228.36,0.000,0.000,0.065,0.065,10.07,154.92,-0.164,-0.164,-5.54,33.78,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.708,23.91,33.77,-2.897,-221.34,76.40,0.0,0.0,-0.016,1.632,-1.48,95.56,7.807,-7.115,0.0,0.0,0.0,0.0,10.07,-5.54,0.065,-0.164,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
4,2025-12-01 00:45:00,1.103,-4.410,64.00,-261.33,0.000,0.000,0.100,0.100,15.45,154.50,-1.151,-0.988,-33.26,33.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.215,7.24,33.67,-3.422,-228.07,66.65,0.0,0.0,0.000,0.788,0.00,41.31,13.123,-9.672,0.0,0.0,0.0,0.0,15.45,-33.26,0.100,-0.988,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
5,2025-12-01 00:45:00,1.103,-4.410,64.00,-261.33,0.000,0.000,0.100,0.100,15.45,154.50,-1.151,-0.988,-33.26,33.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.215,7.24,33.67,-3.422,-228.07,66.65,0.0,0.0,0.000,0.788,0.00,41.31,13.123,-9.672,0.0,0.0,0.0,0.0,15.45,-33.26,0.100,-0.988,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
6,2025-12-01 01:00:00,1.455,-4.339,122.34,-185.75,0.000,0.000,0.000,0.000,0.00,0.00,-1.958,-1.760,-59.38,33.74,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.256,10.87,42.46,-2.523,-126.37,50.09,0.0,0.0,-0.056,1.199,0.00,111.47,13.150,-10.074,0.0,0.0,0.0,0.0,0.00,-59.38,0.000,-1.760,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
7,2025-12-01 01:00:00,1.455,-4.339,122.34,-185.75,0.000,0.000,0.000,0.000,0.00,0.00,-1.958,-1.760,-59.38,33.74,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.256,10.87,42.46,-2.523,-126.37,50.09,0.0,0.0,-0.056,1.199,0.00,111.47,1

In [13]:
# Show settlement type counts for datasets that contain settlement_type.
# This is important for Borzen imbalance and balancing data.

for dataset_name, df in datasets.items():
    if "settlement_type" in df.columns:
        print("=" * 100)
        print(dataset_name)

        settlement_counts = (
            df["settlement_type"]
            .value_counts(dropna=False)
            .rename_axis("settlement_type")
            .reset_index(name="rows")
        )

        display(settlement_counts)

imbalance


,settlement_type,rows
0,Second imbalance settlement,8832
1,First imbalance settlement,2688


balancing


,settlement_type,rows
0,Second imbalance settlement,14784
1,First imbalance settlement,5376


## Duplicate and settlement-type finding

Duplicate timestamps must be interpreted by dataset.

For DA 2025, duplicate local timestamps are expected around the autumn DST transition, when the clock moves back and the 02:00 hour repeats.

For Borzen datasets, timestamp alone may not always be the correct uniqueness key because the data includes `settlement_type`.

Therefore:
- DA duplicate timestamps are checked against DST behavior.
- Borzen duplicate timestamps are checked against `timestamp + settlement_type`.

In [14]:
# Add MTU metadata to each dataset based on configured MTU rules.
# This is necessary because DA 2025 has mixed 60-minute and 15-minute resolution.

datasets_with_mtu = {}

for dataset_name, df in datasets.items():
    config = DATASET_CONFIG[dataset_name]

    df_mtu = df.copy()
    df_mtu["mtu_minutes"] = assign_mtu_minutes(
        df_mtu,
        mtu_rules=config["mtu_rules"],
        timestamp_col="timestamp"
    )

    df_mtu["delivery_date"] = df_mtu["timestamp"].dt.date

    datasets_with_mtu[dataset_name] = df_mtu

    print(dataset_name)
    display(
        df_mtu["mtu_minutes"]
        .value_counts(dropna=False)
        .rename_axis("mtu_minutes")
        .reset_index(name="rows")
    )

da_2025


,mtu_minutes,rows
0,15,8836
1,60,6551


da_2026


,mtu_minutes,rows
0,15,8541


imbalance


,mtu_minutes,rows
0,15,11520


balancing


,mtu_minutes,rows
0,15,20160


In [26]:
# Count expected intervals per delivery day and compare them with actual data.
# This is the core completeness check for Day 22.

daily_completeness_results = []

for dataset_name, df in datasets_with_mtu.items():
    df_check = df.copy()

    # Count raw rows per delivery day and MTU.
    # For DA and imbalance data, one row normally represents one market interval.
    row_counts = (
        df_check
        .groupby(["delivery_date", "mtu_minutes"])
        .size()
        .reset_index(name="row_count")
    )

    # Count unique timestamp labels per delivery day and MTU.
    # This is useful for datasets where one timestamp can appear multiple times,
    # for example balancing data with different settlement_type values.
    unique_timestamp_counts = (
        df_check
        .groupby(["delivery_date", "mtu_minutes"])["timestamp"]
        .nunique()
        .reset_index(name="unique_timestamp_count")
    )

    # Combine row count and unique timestamp count into one QA table.
    daily_counts = row_counts.merge(
        unique_timestamp_counts,
        on=["delivery_date", "mtu_minutes"],
        how="left"
    )

    daily_counts["dataset"] = dataset_name

    # Calculate expected market intervals based on MTU and DST day type.
    # 60-min normal day = 24
    # 60-min spring DST day = 23
    # 60-min autumn DST day = 25
    # 15-min normal day = 96
    # 15-min spring DST day = 92
    # 15-min autumn DST day = 100
    daily_counts["expected_intervals"] = daily_counts.apply(
        lambda row: expected_market_intervals(
            row["delivery_date"],
            int(row["mtu_minutes"])
        ),
        axis=1
    )

    # Add DST classification for easier interpretation.
    daily_counts["dst_day_type"] = daily_counts["delivery_date"].apply(dst_day_type)

    # Choose the correct actual interval count basis.
    # DA and imbalance: use row_count because each row is one market interval.
    # Balancing: use unique_timestamp_count because duplicate rows can exist due to settlement_type.
    if dataset_name == "balancing":
        daily_counts["actual_interval_count"] = daily_counts["unique_timestamp_count"]
        daily_counts["count_basis"] = "unique_timestamp_count"
    else:
        daily_counts["actual_interval_count"] = daily_counts["row_count"]
        daily_counts["count_basis"] = "row_count"

    # Compare actual interval count with expected interval count.
    daily_counts["is_complete_day"] = (
        daily_counts["actual_interval_count"] == daily_counts["expected_intervals"]
    )

    daily_completeness_results.append(daily_counts)

# Combine all dataset-level daily completeness checks into one table.
daily_completeness_df = (
    pd.concat(daily_completeness_results, ignore_index=True)
    .sort_values(["dataset", "delivery_date"])
    .reset_index(drop=True)
)

display(daily_completeness_df)

,delivery_date,mtu_minutes,row_count,unique_timestamp_count,dataset,expected_intervals,dst_day_type,actual_interval_count,count_basis,is_complete_day
0,2025-11-01,15,95,95,balancing,96,normal_day,95,unique_timestamp_count,False
1,2025-11-02,15,96,96,balancing,96,normal_day,96,unique_timestamp_count,True
2,2025-11-03,15,96,96,balancing,96,normal_day,96,unique_timestamp_count,True
3,2025-11-04,15,96,96,balancing,96,normal_day,96,unique_timestamp_count,True
4,2025-11-05,15,96,96,balancing,96,normal_day,96,unique_timestamp_count,True
...,...,...,...,...,...,...,...,...,...,...
693,2026-02-25,15,96,96,imbalance,96,normal_day,96,row_count,True
694,2026-02-26,15,96,96,imbalance,96,normal_day,96,row_count,True
695,2026-02-27,15,96,96,imbalance,96,normal_day,96,row_count,True
696,2026-02-28,15,96,96,imbalance,96,normal_day,96,row_count,True


In [27]:
# Show all days where interval count does not match the expected count.
# These are the main completeness findings for Day 22.

incomplete_days = daily_completeness_df[
    daily_completeness_df["is_complete_day"] == False
].copy()

display(incomplete_days)

,delivery_date,mtu_minutes,row_count,unique_timestamp_count,dataset,expected_intervals,dst_day_type,actual_interval_count,count_basis,is_complete_day
0,2025-11-01,15,95,95,balancing,96,normal_day,95,unique_timestamp_count,False
120,2026-03-01,15,2,1,balancing,96,normal_day,1,unique_timestamp_count,False
485,2025-12-31,15,95,95,da_2025,96,normal_day,95,row_count,False
486,2026-01-01,15,1,1,da_2025,96,normal_day,1,row_count,False
576,2026-04-01,15,1,1,da_2026,96,normal_day,1,row_count,False
577,2025-11-01,15,95,95,imbalance,96,normal_day,95,row_count,False
697,2026-03-01,15,1,1,imbalance,96,normal_day,1,row_count,False


In [29]:
# Add boundary-day classification to separate partial file edges from true internal gaps.
# A partial first or last day is usually less serious than a missing interval inside the full coverage period.

dataset_boundaries = []

for dataset_name, df in datasets_with_mtu.items():
    dataset_boundaries.append({
        "dataset": dataset_name,
        "first_delivery_date": df["delivery_date"].min(),
        "last_delivery_date": df["delivery_date"].max(),
    })

dataset_boundaries_df = pd.DataFrame(dataset_boundaries)

daily_completeness_df = daily_completeness_df.merge(
    dataset_boundaries_df,
    on="dataset",
    how="left"
)

daily_completeness_df["is_boundary_day"] = (
    (daily_completeness_df["delivery_date"] == daily_completeness_df["first_delivery_date"]) |
    (daily_completeness_df["delivery_date"] == daily_completeness_df["last_delivery_date"])
)

daily_completeness_df["qa_status"] = np.select(
    [
        daily_completeness_df["is_complete_day"],
        (~daily_completeness_df["is_complete_day"]) & daily_completeness_df["is_boundary_day"],
        (~daily_completeness_df["is_complete_day"]) & (~daily_completeness_df["is_boundary_day"]),
    ],
    [
        "complete",
        "partial_boundary_day",
        "internal_incomplete_day",
    ],
    default="review"
)

display(daily_completeness_df)

,delivery_date,mtu_minutes,row_count,unique_timestamp_count,dataset,expected_intervals,dst_day_type,actual_interval_count,count_basis,is_complete_day,first_delivery_date,last_delivery_date,is_boundary_day,qa_status
0,2025-11-01,15,95,95,balancing,96,normal_day,95,unique_timestamp_count,False,2025-11-01,2026-03-01,True,partial_boundary_day
1,2025-11-02,15,96,96,balancing,96,normal_day,96,unique_timestamp_count,True,2025-11-01,2026-03-01,False,complete
2,2025-11-03,15,96,96,balancing,96,normal_day,96,unique_timestamp_count,True,2025-11-01,2026-03-01,False,complete
3,2025-11-04,15,96,96,balancing,96,normal_day,96,unique_timestamp_count,True,2025-11-01,2026-03-01,False,complete
4,2025-11-05,15,96,96,balancing,96,normal_day,96,unique_timestamp_count,True,2025-11-01,2026-03-01,False,complete
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
693,2026-02-25,15,96,96,imbalance,96,normal_day,96,row_count,True,2025-11-01,2026-03-01,False,complete
694,2026-02-26,15,96,96,imbalance,96,normal_day,96,row_count,True,2025-11-01,2026-03-01,False,complete
695,2026-02-27,15,96,96,imbalance,96,normal_day,96,row_count,True,2025-11-01,2026-03-01,False,complete
696,2026-02-28,15,96,96,imbalance,96,normal_day,96,row_count,True,2025-11-01,2026-03-01,False,complete


In [30]:
# Show only non-complete days after boundary classification.
# Internal incomplete days are the most important QA problems.

qa_problem_days = daily_completeness_df[
    daily_completeness_df["qa_status"] != "complete"
].copy()

display(
    qa_problem_days[
        [
            "dataset",
            "delivery_date",
            "mtu_minutes",
            "row_count",
            "unique_timestamp_count",
            "actual_interval_count",
            "expected_intervals",
            "count_basis",
            "dst_day_type",
            "is_boundary_day",
            "qa_status",
        ]
    ]
)

,dataset,delivery_date,mtu_minutes,row_count,unique_timestamp_count,actual_interval_count,expected_intervals,count_basis,dst_day_type,is_boundary_day,qa_status
0,balancing,2025-11-01,15,95,95,95,96,unique_timestamp_count,normal_day,True,partial_boundary_day
120,balancing,2026-03-01,15,2,1,1,96,unique_timestamp_count,normal_day,True,partial_boundary_day
485,da_2025,2025-12-31,15,95,95,95,96,row_count,normal_day,False,internal_incomplete_day
486,da_2025,2026-01-01,15,1,1,1,96,row_count,normal_day,True,partial_boundary_day
576,da_2026,2026-04-01,15,1,1,1,96,row_count,normal_day,True,partial_boundary_day
577,imbalance,2025-11-01,15,95,95,95,96,row_count,normal_day,True,partial_boundary_day
697,imbalance,2026-03-01,15,1,1,1,96,row_count,normal_day,True,partial_boundary_day


In [28]:
# Inspect DST transition days directly.
# This confirms whether March and October DST behavior is handled correctly.

dst_days_to_check = [
    pd.Timestamp("2025-03-30").date(),
    pd.Timestamp("2025-10-26").date(),
    pd.Timestamp("2026-03-29").date(),
    pd.Timestamp("2026-10-25").date(),
]

dst_check = daily_completeness_df[
    daily_completeness_df["delivery_date"].isin(dst_days_to_check)
].copy()

display(dst_check)

,delivery_date,mtu_minutes,row_count,unique_timestamp_count,dataset,expected_intervals,dst_day_type,actual_interval_count,count_basis,is_complete_day
209,2025-03-30,60,23,23,da_2025,23,spring_dst_day,23,row_count,True
419,2025-10-26,15,100,96,da_2025,100,autumn_dst_day,100,row_count,True
573,2026-03-29,15,92,92,da_2026,92,spring_dst_day,92,row_count,True


In [18]:
# Show timestamps for the 2025 spring DST day in DA 2025.
# Because DA 2025 is hourly in March, we expect 23 rows and no 02:00 timestamp.

da_2025 = datasets_with_mtu["da_2025"]

spring_2025 = (
    da_2025[da_2025["delivery_date"] == pd.Timestamp("2025-03-30").date()]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

spring_2025_times = spring_2025[["timestamp", "price_eur_mwh", "mtu_minutes"]].copy()
spring_2025_times["clock_time"] = spring_2025_times["timestamp"].dt.strftime("%H:%M")

print("Rows on 2025-03-30:", len(spring_2025_times))
display(spring_2025_times)

Rows on 2025-03-30: 23


,timestamp,price_eur_mwh,mtu_minutes,clock_time
0,2025-03-30 00:00:00,46.22,60,00:00
1,2025-03-30 01:00:00,15.88,60,01:00
2,2025-03-30 03:00:00,5.09,60,03:00
3,2025-03-30 04:00:00,1.20,60,04:00
4,2025-03-30 05:00:00,0.09,60,05:00
5,2025-03-30 06:00:00,0.93,60,06:00
6,2025-03-30 07:00:00,0.99,60,07:00
7,2025-03-30 08:00:00,1.45,60,08:00
8,2025-03-30 09:00:00,0.09,60,09:00
9,2025-03-30 10:00:00,-0.76,60,10:00


In [19]:
# Show duplicate timestamps on the 2025 autumn DST day in DA 2025.
# Because the clock moves back, the 02:00 hour repeats in local time.

autumn_2025 = (
    da_2025[da_2025["delivery_date"] == pd.Timestamp("2025-10-26").date()]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

autumn_2025_duplicates = (
    autumn_2025[autumn_2025["timestamp"].duplicated(keep=False)]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("Rows on 2025-10-26:", len(autumn_2025))
print("Duplicate timestamp rows on 2025-10-26:", len(autumn_2025_duplicates))

display(autumn_2025_duplicates)

Rows on 2025-10-26: 100
Duplicate timestamp rows on 2025-10-26: 8


,timestamp,price_eur_mwh,mtu_minutes,delivery_date
0,2025-10-26 02:00:00,75.33,15,2025-10-26
1,2025-10-26 02:00:00,63.05,15,2025-10-26
2,2025-10-26 02:15:00,65.54,15,2025-10-26
3,2025-10-26 02:15:00,67.41,15,2025-10-26
4,2025-10-26 02:30:00,60.01,15,2025-10-26
5,2025-10-26 02:30:00,63.93,15,2025-10-26
6,2025-10-26 02:45:00,53.67,15,2025-10-26
7,2025-10-26 02:45:00,58.91,15,2025-10-26


## DST findings

The DA 2025 file correctly reflects the spring DST transition on 2025-03-30.  
Because the file is hourly in March 2025, the expected count is 23 intervals instead of 24.

The DA 2025 file also contains duplicate local timestamps around the autumn DST transition on 2025-10-26.  
This is expected because the clock moves back from 03:00 to 02:00 and the 02:00 hour repeats.

For 15-minute data:
- normal day = 96 intervals,
- spring DST day = 92 intervals,
- autumn DST day = 100 intervals.

DST-related duplicates are not automatically data-quality errors.

In [20]:
# Identify missing timestamp occurrences for incomplete days.
# This is useful when a daily count is lower than expected.

missing_interval_results = []

for dataset_name, df in datasets_with_mtu.items():
    config = DATASET_CONFIG[dataset_name]
    count_method = config["interval_count_method"]

    dataset_incomplete_days = incomplete_days[
        incomplete_days["dataset"] == dataset_name
    ]

    for _, day_row in dataset_incomplete_days.iterrows():
        delivery_date = day_row["delivery_date"]
        mtu_minutes = int(day_row["mtu_minutes"])

        day_data = df[df["delivery_date"] == delivery_date].copy()

        expected_counts = expected_timestamp_occurrences_for_day(delivery_date, mtu_minutes)

        if count_method == "rows":
            actual_counts = (
                day_data
                .value_counts("timestamp")
                .reset_index(name="actual_occurrences")
            )
        else:
            actual_counts = (
                day_data[["timestamp"]]
                .drop_duplicates()
                .value_counts("timestamp")
                .reset_index(name="actual_occurrences")
            )

        comparison = expected_counts.merge(
            actual_counts,
            on="timestamp",
            how="left"
        )

        comparison["actual_occurrences"] = comparison["actual_occurrences"].fillna(0).astype(int)
        comparison["missing_occurrences"] = comparison["expected_occurrences"] - comparison["actual_occurrences"]

        missing = comparison[comparison["missing_occurrences"] > 0].copy()
        missing["dataset"] = dataset_name
        missing["delivery_date"] = delivery_date
        missing["mtu_minutes"] = mtu_minutes
        missing["count_method"] = count_method

        missing_interval_results.append(missing)

if missing_interval_results:
    missing_intervals_df = (
        pd.concat(missing_interval_results, ignore_index=True)
        .sort_values(["dataset", "timestamp"])
        .reset_index(drop=True)
    )
else:
    missing_intervals_df = pd.DataFrame()

display(missing_intervals_df)

,timestamp,expected_occurrences,actual_occurrences,missing_occurrences,dataset,delivery_date,mtu_minutes,count_method
0,2025-11-01 00:00:00,1,0,1,balancing,2025-11-01,15,unique_timestamps
1,2026-03-01 00:15:00,1,0,1,balancing,2026-03-01,15,unique_timestamps
2,2026-03-01 00:30:00,1,0,1,balancing,2026-03-01,15,unique_timestamps
3,2026-03-01 00:45:00,1,0,1,balancing,2026-03-01,15,unique_timestamps
4,2026-03-01 01:00:00,1,0,1,balancing,2026-03-01,15,unique_timestamps
...,...,...,...,...,...,...,...,...
378,2026-03-01 22:45:00,1,0,1,imbalance,2026-03-01,15,rows
379,2026-03-01 23:00:00,1,0,1,imbalance,2026-03-01,15,rows
380,2026-03-01 23:15:00,1,0,1,imbalance,2026-03-01,15,rows
381,2026-03-01 23:30:00,1,0,1,imbalance,2026-03-01,15,rows


In [21]:
# Combine DA 2025 and DA 2026 data for overlap checks with imbalance data.
# Source file is kept so boundary overlaps can be investigated.

da_all = pd.concat(
    [
        datasets["da_2025"].assign(source_file="da_prices_si_2025_clean.csv"),
        datasets["da_2026"].assign(source_file="da_prices_si_clean.csv"),
    ],
    ignore_index=True
)

da_all = da_all.sort_values("timestamp").reset_index(drop=True)

# Drop duplicate DA timestamps for join checks.
# keep="last" gives priority to the 2026 file where the annual boundary overlaps.
da_unique = da_all.drop_duplicates(subset=["timestamp"], keep="last").copy()

imbalance = datasets["imbalance"].copy()

# Find imbalance timestamps that are not present in DA.
unmatched_imbalance = imbalance[
    ~imbalance["timestamp"].isin(da_unique["timestamp"])
].copy()

print("DA all rows:", len(da_all))
print("DA unique timestamp rows:", len(da_unique))
print("Imbalance rows:", len(imbalance))
print("Unmatched imbalance rows:", len(unmatched_imbalance))

display(unmatched_imbalance)

DA all rows: 23928
DA unique timestamp rows: 23923
Imbalance rows: 11520
Unmatched imbalance rows: 1


,timestamp,imbalance_price_pos,imbalance_price_neg,sipx,settlement_type
5759,2025-12-31,112.79,112.79,77.8,Second imbalance settlement


## DA + imbalance timestamp mismatch

The DA and imbalance datasets mostly align in the overlapping period.

A previous cross-dataset check showed:
- 11,519 DA rows in the overlap,
- 11,520 imbalance rows,
- 11,519 joined rows.

The one unmatched imbalance row is caused by a missing DA timestamp.

This is a real QA finding because it affects cross-dataset analysis.  
Any DA + imbalance analysis should either exclude this unmatched row or clearly document that the analysis uses matched timestamps only.

In [22]:
# Check important numeric columns for negative values, zeros, and high values.
# Negative prices and spikes can be valid in power markets, so this flags values for review, not automatic deletion.

value_check_results = []

for dataset_name, df in datasets.items():
    config = DATASET_CONFIG[dataset_name]

    columns_to_check = []
    columns_to_check.extend(config.get("price_columns", []))
    columns_to_check.extend(config.get("important_columns", []))

    # Keep only numeric columns that exist in the dataset.
    columns_to_check = [
        col for col in dict.fromkeys(columns_to_check)
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col])
    ]

    for col in columns_to_check:
        value_check_results.append({
            "dataset": dataset_name,
            "column": col,
            "rows": len(df),
            "min": df[col].min(),
            "p01": df[col].quantile(0.01),
            "mean": df[col].mean(),
            "median": df[col].median(),
            "p99": df[col].quantile(0.99),
            "max": df[col].max(),
            "negative_count": (df[col] < 0).sum(),
            "zero_count": (df[col] == 0).sum(),
            "above_500_count": (df[col] > 500).sum(),
        })

value_checks_df = pd.DataFrame(value_check_results)

display(value_checks_df)

,dataset,column,rows,min,p01,mean,median,p99,max,negative_count,zero_count,above_500_count
0,da_2025,price_eur_mwh,15387,-126.460,-2.0000,108.266467,105.2100,270.77840,561.750,269,62,5
1,da_2026,price_eur_mwh,8541,-69.920,0.0000,121.055885,118.0000,278.30400,400.760,84,12,0
2,imbalance,imbalance_price_pos,11520,-3285.020,-10.7710,121.935337,111.8100,375.79870,2745.440,204,5,36
3,imbalance,imbalance_price_neg,11520,-3285.020,-10.7710,121.935337,111.8100,375.79870,2745.440,204,5,36
4,imbalance,sipx,11520,-0.440,43.4319,120.018549,112.5900,277.16340,400.760,14,5,0
5,balancing,aFRR EUR/MWh,20160,0.000,0.0000,149.266044,167.8500,392.29000,579.330,0,4974,40
6,balancing,mFRR EUR/MWh,20160,0.000,0.0000,3.692448,0.0000,0.00000,3293.270,0,20124,36
7,balancing,VoAA+ EUR/MWh,20160,0.000,0.0000,0.015704,0.0000,0.00000,158.300,0,20158,0
8,balancing,VoAA- EUR/MWh,20160,0.000,0.0000,0.008879,0.0000,0.00000,89.500,0,20158,0
9,balancing,Wpos MWh,20160,0.000,0.0270,10.144370,5.8090,59.63558,135.782,0,113,0


In [23]:
# Show rows where price-like columns are negative, zero, or above 500.
# This is used for inspection only; these values may be valid market outcomes.

extreme_price_rows = []

for dataset_name, df in datasets.items():
    config = DATASET_CONFIG[dataset_name]

    for col in config.get("price_columns", []):
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            flagged = df[
                (df[col] < 0) |
                (df[col] == 0) |
                (df[col] > 500)
            ].copy()

            if len(flagged) > 0:
                flagged["dataset"] = dataset_name
                flagged["checked_column"] = col
                flagged["checked_value"] = flagged[col]

                keep_cols = ["dataset", "timestamp", "checked_column", "checked_value"]

                if "settlement_type" in flagged.columns:
                    keep_cols.append("settlement_type")

                extreme_price_rows.append(flagged[keep_cols])

if extreme_price_rows:
    extreme_price_rows_df = (
        pd.concat(extreme_price_rows, ignore_index=True)
        .sort_values(["dataset", "timestamp", "checked_column"])
        .reset_index(drop=True)
    )
else:
    extreme_price_rows_df = pd.DataFrame()

display(extreme_price_rows_df.head(200))
print("Total flagged price rows:", len(extreme_price_rows_df))

,dataset,timestamp,checked_column,checked_value,settlement_type
0,balancing,2025-11-01 00:15:00,VoAA+ EUR/MWh,0.0,Second imbalance settlement
1,balancing,2025-11-01 00:15:00,VoAA- EUR/MWh,0.0,Second imbalance settlement
2,balancing,2025-11-01 00:15:00,mFRR EUR/MWh,0.0,Second imbalance settlement
3,balancing,2025-11-01 00:30:00,VoAA+ EUR/MWh,0.0,Second imbalance settlement
4,balancing,2025-11-01 00:30:00,VoAA- EUR/MWh,0.0,Second imbalance settlement
5,balancing,2025-11-01 00:30:00,aFRR EUR/MWh,0.0,Second imbalance settlement
6,balancing,2025-11-01 00:30:00,mFRR EUR/MWh,0.0,Second imbalance settlement
7,balancing,2025-11-01 00:45:00,VoAA+ EUR/MWh,0.0,Second imbalance settlement
8,balancing,2025-11-01 00:45:00,VoAA- EUR/MWh,0.0,Second imbalance settlement
9,balancing,2025-11-01 00:45:00,aFRR EUR/MWh,0.0,Second imbalance settlement


Total flagged price rows: 66431


## Value range findings

The value range check flags:
- negative values,
- zero values,
- values above 500.

For DA and imbalance prices, negative prices and spikes can be valid electricity-market outcomes.  
They are not automatically data errors.

For balancing volumes and costs, zeros are common because not every balancing product is activated in every interval.  
Negative values may also be meaningful depending on direction and settlement logic.

Outlier treatment should be handled separately in a later analysis step.

In [24]:
# Build a final compact QA summary table for Day 22.
# This gives a high-level status per dataset.

final_qa_summary = []

for dataset_name, df in datasets.items():
    dataset_incomplete = daily_completeness_df[
        daily_completeness_df["dataset"] == dataset_name
    ]

    final_qa_summary.append({
        "dataset": dataset_name,
        "rows": len(df),
        "first_timestamp": df["timestamp"].min(),
        "last_timestamp": df["timestamp"].max(),
        "null_values": df.isna().sum().sum(),
        "duplicate_timestamps": df["timestamp"].duplicated().sum(),
        "incomplete_days": (dataset_incomplete["is_complete_day"] == False).sum(),
        "spring_dst_days_checked": (dataset_incomplete["dst_day_type"] == "spring_dst_day").sum(),
        "autumn_dst_days_checked": (dataset_incomplete["dst_day_type"] == "autumn_dst_day").sum(),
    })

final_qa_summary_df = pd.DataFrame(final_qa_summary)

display(final_qa_summary_df)

,dataset,rows,first_timestamp,last_timestamp,null_values,duplicate_timestamps,incomplete_days,spring_dst_days_checked,autumn_dst_days_checked
0,da_2025,15387,2025-01-01 00:00:00,2026-01-01,0,4,2,1,1
1,da_2026,8541,2026-01-01 00:00:00,2026-04-01,0,0,1,1,0
2,imbalance,11520,2025-11-01 00:15:00,2026-03-01,0,0,2,0,0
3,balancing,20160,2025-11-01 00:15:00,2026-03-01,0,8640,2,0,0


## Day 22 QA conclusion

The curated datasets are structurally usable for analysis.  
Timestamps are parsed correctly and the loaded datasets contain no null values.

Important findings:

1. The 2025 DA file has mixed MTU resolution:
   - 60-minute before 2025-10-01,
   - 15-minute from 2025-10-01 onward.
   This is expected and reflects the Slovenian day-ahead market-rule transition.

2. DST checks must be resolution-aware:
   - normal hourly day: 24 intervals,
   - spring DST hourly day: 23 intervals,
   - autumn DST hourly day: 25 intervals,
   - normal 15-minute day: 96 intervals,
   - spring DST 15-minute day: 92 intervals,
   - autumn DST 15-minute day: 100 intervals.

3. DA 2025 duplicate timestamps are linked to autumn DST repeated local timestamps and should not be removed automatically.

4. Borzen imbalance and balancing data must be interpreted with `settlement_type`.

5. The DA + imbalance overlap has one unmatched imbalance timestamp caused by a missing DA timestamp.

6. Boundary days and incomplete days must be documented before using daily summaries or joins.

7. Negative prices, zeros, and spikes are flagged for review, but they are not automatically data errors in electricity-market data.

Overall, the data is suitable for continued analysis if future joins, daily summaries, and visualizations remain MTU-aware, DST-aware, and explicit about incomplete periods.